# Qwen2.5-1.5B-Instruct PyTorch: Mermaid dan Fine-Tuning dari Web UI

Notebook ini memakai **Qwen/Qwen2.5-1.5B-Instruct** dalam format Hugging Face Transformers/PyTorch (`safetensors`), bukan GGUF.

Fitur utama:

1. Inferensi Mermaid memakai `transformers` dan PyTorch.
2. Aplikasi web menyediakan tab **Generator Mermaid**.
3. Aplikasi web menyediakan tab **Dataset & Fine-Tuning** untuk mengunggah dan memvalidasi dataset.
4. Fine-tuning dapat dijalankan dari browser dalam mode **LoRA**, **QLoRA 4-bit**, atau **Full Fine-Tuning**.
5. Progres, loss, log training, pembatalan, dan hasil training ditampilkan melalui UI.
6. Adapter/model hasil training otomatis menjadi model aktif untuk inferensi Mermaid.
7. Adapter hasil LoRA/QLoRA dapat diunduh sebagai ZIP.

Format dataset yang diterima:

- JSONL/JSON dengan kolom `messages`.
- Pasangan `prompt`–`completion`.
- Pasangan `instruction`–`output`, dengan `input` opsional.

> **Kompatibilitas:** notebook mengunci `peft==0.18.1` untuk menghindari regresi dispatcher AWQ/GPTQModel pada PEFT 0.19.x.


In [10]:
# 1) Instalasi library PyTorch/Transformers yang kompatibel
#
# PENTING:
# PEFT 0.19.x memiliki regresi pada dispatcher AWQ/GPTQModel yang dapat
# menyebabkan ImportError saat get_peft_model() dipanggil, bahkan ketika
# model yang digunakan bukan model AWQ. Notebook ini mengunci PEFT 0.18.1.
#
# Bila sel ini mengganti versi library yang sudah pernah diimpor pada sesi
# berjalan, runtime harus dimulai ulang agar Python tidak memakai modul lama.

import importlib.metadata
import os
import shutil
import signal
import subprocess
import sys
import time

PINNED_PACKAGES = [
    # Transformers 4.x stabil dan mendukung arsitektur Qwen2.5.
    "transformers>=4.52,<5",
    # Hindari regresi dispatcher AWQ pada PEFT 0.19.x.
    "peft==0.18.1",
    "accelerate>=1.2,<2",
    "datasets>=3,<5",
    "bitsandbytes>=0.44,<1",
    "safetensors>=0.4",
    "sentencepiece",
    "flask",
    "werkzeug",
    "requests",
    "packaging",
]

TRACKED_MODULES = {
    "transformers",
    "peft",
    "accelerate",
    "datasets",
    "bitsandbytes",
}

def installed_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None

versions_before = {
    name: installed_version(name)
    for name in TRACKED_MODULES
}

modules_loaded_before_install = sorted(
    name for name in TRACKED_MODULES if name in sys.modules
)

print("Versi sebelum instalasi:")
for name in sorted(versions_before):
    print(f"  {name}: {versions_before[name] or 'belum terpasang'}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        *PINNED_PACKAGES,
    ],
    check=True,
)

# Pastikan metadata distribusi dibaca kembali setelah pip selesai.
importlib.invalidate_caches()

versions_after = {
    name: installed_version(name)
    for name in TRACKED_MODULES
}

print("\nVersi setelah instalasi:")
for name in sorted(versions_after):
    print(f"  {name}: {versions_after[name] or 'tidak ditemukan'}")

if versions_after.get("peft") != "0.18.1":
    raise RuntimeError(
        "PEFT 0.18.1 gagal dipasang. Versi yang terdeteksi: "
        f"{versions_after.get('peft')}"
    )

changed_loaded_modules = [
    name
    for name in modules_loaded_before_install
    if versions_before.get(name) != versions_after.get(name)
]

if changed_loaded_modules:
    message = (
        "Versi library yang sudah aktif di memori telah berubah: "
        + ", ".join(changed_loaded_modules)
        + ". Runtime perlu dimulai ulang sebelum melanjutkan."
    )
    print("\n" + "=" * 72)
    print(message)
    print("=" * 72)

    # Colab: mulai ulang sesi secara otomatis untuk mencegah campuran versi.
    try:
        import google.colab  # noqa: F401

        print("Runtime Colab akan dimulai ulang otomatis.")
        print("Setelah tersambung kembali, jalankan notebook dari awal.")
        time.sleep(2)
        os.kill(os.getpid(), signal.SIGKILL)
    except ImportError:
        raise RuntimeError(
            message
            + " Gunakan menu Kernel/Runtime > Restart, lalu jalankan notebook dari awal."
        )

# Baru impor library setelah versi akhir dipastikan benar.
import torch
import transformers
import accelerate
import datasets
import peft

print("\nLingkungan siap:")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

if peft.__version__ != "0.18.1":
    raise RuntimeError(
        "Modul PEFT yang aktif bukan 0.18.1. Restart runtime lalu jalankan ulang."
    )

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime dari PyTorch:", torch.version.cuda)
    print("Compute capability:", torch.cuda.get_device_capability(0))
else:
    print(
        "PERINGATAN: GPU CUDA tidak ditemukan. Inferensi CPU tetap dapat "
        "dijalankan, tetapi fine-tuning akan sangat lambat."
    )

# gptqmodel tidak diperlukan untuk Qwen PyTorch biasa maupun QLoRA bitsandbytes.
gptqmodel_version = installed_version("gptqmodel")
if gptqmodel_version:
    print(
        "Catatan: gptqmodel terpasang dengan versi",
        gptqmodel_version,
        "tetapi tidak digunakan oleh notebook ini.",
    )

print("cloudflared:", shutil.which("cloudflared") or "akan dipasang saat aplikasi dijalankan")


Versi sebelum instalasi:
  accelerate: 1.14.0
  bitsandbytes: 0.49.2
  datasets: 4.8.5
  peft: 0.18.1
  transformers: 4.57.6

Versi setelah instalasi:
  accelerate: 1.14.0
  bitsandbytes: 0.49.2
  datasets: 4.8.5
  peft: 0.18.1
  transformers: 4.57.6

Lingkungan siap:
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Accelerate: 1.14.0
Datasets: 4.8.5
PEFT: 0.18.1
GPU: Tesla T4
CUDA runtime dari PyTorch: 12.8
Compute capability: (7, 5)
cloudflared: /usr/local/bin/cloudflared


In [11]:
# 2) Memuat Qwen2.5-1.5B-Instruct versi PyTorch/Hugging Face
#
# USE_4BIT = False:
#   Model dimuat dalam BF16/FP16 dan cocok untuk LoRA maupun full fine-tuning.
#
# USE_4BIT = True:
#   Model tetap merupakan model PyTorch/Hugging Face, tetapi bobot dasarnya
#   dimuat 4-bit memakai bitsandbytes agar dapat dilakukan QLoRA.
#
# LOAD_ADAPTER_PATH:
#   Isi dengan direktori adapter LoRA untuk memakai hasil fine-tuning yang
#   pernah disimpan. Biarkan None untuk memuat model dasar.

from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = False
LOAD_ADAPTER_PATH = None  # opsional; UI fine-tuning dapat mengganti model aktif

if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError("USE_4BIT=True memerlukan GPU CUDA yang didukung bitsandbytes.")

if torch.cuda.is_available():
    COMPUTE_DTYPE = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
else:
    COMPUTE_DTYPE = torch.float32

qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=False,
)

if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_tokenizer.padding_side = "right"

model_kwargs = {
    "torch_dtype": COMPUTE_DTYPE,
    "low_cpu_mem_usage": True,
    "trust_remote_code": False,
}

if torch.cuda.is_available():
    # Satu GPU Colab. Penempatan eksplisit ini juga aman untuk proses training.
    model_kwargs["device_map"] = {"": 0}

if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **model_kwargs,
)

if LOAD_ADAPTER_PATH:
    adapter_path = Path(LOAD_ADAPTER_PATH)
    if not adapter_path.exists():
        raise FileNotFoundError(f"Adapter tidak ditemukan: {adapter_path}")

    from peft import PeftModel

    qwen_model = PeftModel.from_pretrained(
        qwen_model,
        str(adapter_path),
        is_trainable=False,
    )
    print("Adapter dimuat:", adapter_path)

qwen_model.eval()
qwen_model.config.use_cache = True

parameter_count = sum(parameter.numel() for parameter in qwen_model.parameters())
input_device = qwen_model.get_input_embeddings().weight.device

print("Model:", MODEL_ID)
print("Format: Hugging Face Transformers / PyTorch safetensors")
print("Kuantisasi 4-bit:", USE_4BIT)
print("Compute dtype:", COMPUTE_DTYPE)
print("Perangkat input:", input_device)
print(f"Jumlah parameter yang terlihat: {parameter_count:,}")

# Status model global akan diperbarui oleh backend fine-tuning web.
MODEL_RUNTIME_STATE = {
    "model_id": MODEL_ID,
    "mode": "base-4bit" if USE_4BIT else "base",
    "adapter_name": None,
    "output_dir": None,
    "updated_at": time.time() if "time" in globals() else None,
}


Model: Qwen/Qwen2.5-1.5B-Instruct
Format: Hugging Face Transformers / PyTorch safetensors
Kuantisasi 4-bit: False
Compute dtype: torch.bfloat16
Perangkat input: cuda:0
Jumlah parameter yang terlihat: 1,543,714,304


In [12]:
# 3) Fungsi inferensi PyTorch yang dipakai oleh tes dan aplikasi Mermaid

import torch

def generate_qwen_text(
    messages,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
    repetition_penalty=1.05,
):
    """
    Menghasilkan jawaban Qwen dari daftar pesan berformat:
    [{"role": "system|user|assistant", "content": "..."}]
    """

    if "qwen_model" not in globals() or "qwen_tokenizer" not in globals():
        raise RuntimeError("Jalankan sel pemuatan model terlebih dahulu.")

    prompt_text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = qwen_tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_device = qwen_model.get_input_embeddings().weight.device
    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    do_sample = temperature is not None and temperature > 0

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": qwen_tokenizer.pad_token_id,
        "eos_token_id": qwen_tokenizer.eos_token_id,
        "use_cache": True,
    }

    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = top_p

    qwen_model.eval()
    qwen_model.config.use_cache = True

    with torch.inference_mode():
        generated = qwen_model.generate(
            **model_inputs,
            **generation_kwargs,
        )

    prompt_length = model_inputs["input_ids"].shape[-1]
    generated_tokens = generated[0, prompt_length:]

    return qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


RUN_MODEL_TEST = True

if RUN_MODEL_TEST:
    test_text = generate_qwen_text(
        [
            {
                "role": "system",
                "content": "Jawab sangat singkat.",
            },
            {
                "role": "user",
                "content": "Tulis tepat: Qwen PyTorch siap",
            },
        ],
        max_new_tokens=24,
        temperature=0.0,
    )
    print("Tes model:", test_text)


Tes model: Qwen dan PyTorch siap.


## Aplikasi Web Terpadu

Jalankan sel berikut setelah instalasi, pemuatan model, dan fungsi inferensi selesai. Aplikasi web berjalan pada proses notebook yang sama sehingga dapat:

- menerima dataset melalui browser;
- memvalidasi dan menyimpan dataset secara kanonik;
- memuat ulang model dasar untuk LoRA, QLoRA, atau full fine-tuning;
- menjalankan training pada thread terpisah;
- menampilkan progres dan log secara berkala;
- membatalkan training pada batas langkah berikutnya;
- mengaktifkan hasil training untuk pembuatan Mermaid.

Sel manual fine-tuning yang sebelumnya langsung menjalankan `Trainer.train()` telah diganti dengan fungsi backend, sehingga **Run All** tidak otomatis memulai pelatihan.


In [13]:
# 4) Backend dataset dan fine-tuning untuk aplikasi web

import gc
import inspect
import json
import math
import os
import re
import shutil
import threading
import time
import traceback
import uuid
from collections.abc import Mapping
from pathlib import Path

import torch
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

NOTEBOOK_PATCH_VERSION = "2026-07-13.3"

# ------------------------------------------------------------------
# Workspace
# ------------------------------------------------------------------

DEFAULT_WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
WEB_WORK_DIR = DEFAULT_WORK_DIR / "qwen_mermaid_web"
WEB_DATASET_DIR = WEB_WORK_DIR / "datasets"
WEB_OUTPUT_DIR = WEB_WORK_DIR / "training_outputs"

for directory in (WEB_WORK_DIR, WEB_DATASET_DIR, WEB_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MAX_UPLOAD_BYTES = 100 * 1024 * 1024
MAX_DATASET_EXAMPLES = 200_000
MAX_LOG_LINES = 250

dataset_registry = {}
dataset_registry_lock = threading.Lock()

training_jobs = {}
training_jobs_lock = threading.Lock()

# Satu operasi berat terhadap model pada satu waktu.
model_operation_lock = threading.Lock()
active_training_lock = threading.Lock()
active_training_job_id = None


# ------------------------------------------------------------------
# Dataset validation and persistence
# ------------------------------------------------------------------

def normalize_training_record(record):
    if not isinstance(record, Mapping):
        raise ValueError("Setiap contoh dataset harus berupa objek JSON.")

    messages = record.get("messages")

    if messages is not None:
        if not isinstance(messages, list) or len(messages) < 2:
            raise ValueError("'messages' harus berupa list dengan minimal dua pesan.")

        normalized = []
        allowed_roles = {"system", "user", "assistant"}

        for index, message in enumerate(messages):
            if not isinstance(message, Mapping):
                raise ValueError(f"messages[{index}] harus berupa objek.")

            role = str(message.get("role", "")).strip().lower()
            content = str(message.get("content", "")).strip()

            if role not in allowed_roles:
                raise ValueError(
                    f"Role '{role}' tidak didukung. Gunakan system, user, atau assistant."
                )
            if not content:
                raise ValueError(f"messages[{index}].content tidak boleh kosong.")

            normalized.append({"role": role, "content": content})

        if normalized[-1]["role"] != "assistant":
            raise ValueError("Pesan terakhir harus berperan sebagai assistant.")

        if not any(message["role"] == "user" for message in normalized[:-1]):
            raise ValueError("Contoh harus memiliki setidaknya satu pesan user.")

        return {"messages": normalized}

    if record.get("prompt") is not None and record.get("completion") is not None:
        prompt = str(record["prompt"]).strip()
        completion = str(record["completion"]).strip()

        if not prompt or not completion:
            raise ValueError("prompt dan completion tidak boleh kosong.")

        return {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": completion},
            ]
        }

    if record.get("instruction") is not None and record.get("output") is not None:
        instruction = str(record["instruction"]).strip()
        output = str(record["output"]).strip()
        extra_input = str(record.get("input") or "").strip()

        if not instruction or not output:
            raise ValueError("instruction dan output tidak boleh kosong.")

        user_content = instruction
        if extra_input:
            user_content += f"\n\nInput:\n{extra_input}"

        return {
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": output},
            ]
        }

    raise ValueError(
        "Contoh harus memakai messages, prompt-completion, "
        "atau instruction-output."
    )


def load_records_from_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    records = []

    if suffix == ".jsonl":
        with path.open("r", encoding="utf-8-sig") as handle:
            for line_number, line in enumerate(handle, start=1):
                text = line.strip()
                if not text:
                    continue
                try:
                    records.append(json.loads(text))
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"JSONL tidak valid pada baris {line_number}: {exc.msg}"
                    ) from exc

    elif suffix == ".json":
        with path.open("r", encoding="utf-8-sig") as handle:
            payload = json.load(handle)

        if isinstance(payload, list):
            records = payload
        elif isinstance(payload, dict):
            for key in ("data", "train", "examples", "records"):
                value = payload.get(key)
                if isinstance(value, list):
                    records = value
                    break
            else:
                records = [payload]
        else:
            raise ValueError("JSON harus berupa list atau objek.")

    else:
        raise ValueError("Format file harus .jsonl atau .json.")

    if not records:
        raise ValueError("Dataset tidak berisi contoh.")

    if len(records) > MAX_DATASET_EXAMPLES:
        raise ValueError(
            f"Dataset berisi {len(records):,} contoh; batas UI adalah "
            f"{MAX_DATASET_EXAMPLES:,} contoh."
        )

    normalized = []
    errors = []

    for index, record in enumerate(records):
        try:
            normalized.append(normalize_training_record(record))
        except Exception as exc:
            errors.append(f"Contoh {index + 1}: {exc}")
            if len(errors) >= 20:
                break

    if errors:
        detail = "\n".join(errors)
        if len(records) > len(errors):
            detail += "\nValidasi dihentikan setelah 20 kesalahan pertama."
        raise ValueError(detail)

    return normalized


def build_dataset_validation_report(records):
    """Return a serializable quality report after schema normalization."""
    fingerprints = set()
    duplicate_examples = 0
    mermaid_examples = 0
    flowchart_td_examples = 0
    start_end_examples = 0
    decision_examples = 0
    assistant_lengths = []
    warnings = []

    for record in records:
        messages = record["messages"]
        fingerprint = json.dumps(messages, ensure_ascii=False, sort_keys=True)
        if fingerprint in fingerprints:
            duplicate_examples += 1
        fingerprints.add(fingerprint)

        answer = messages[-1]["content"].strip()
        assistant_lengths.append(len(answer))
        is_mermaid = answer.startswith("```mermaid") and answer.endswith("```")
        mermaid_examples += int(is_mermaid)
        flowchart_td_examples += int(bool(re.search(r"(?m)^\s*flowchart\s+TD\s*$", answer)))
        start_end_examples += int("Mulai" in answer and "Selesai" in answer)
        decision_examples += int("{" in answer and "}" in answer)

    total = len(records)
    duplicate_ratio = duplicate_examples / total
    mermaid_ratio = mermaid_examples / total
    if total < 20:
        warnings.append("Dataset sangat kecil; disarankan minimal 50-100 contoh berkualitas.")
    if duplicate_ratio > 0:
        warnings.append(f"Terdapat {duplicate_examples} contoh duplikat persis ({duplicate_ratio:.1%}).")
    if 0 < mermaid_examples < total:
        warnings.append("Format output bercampur: hanya sebagian jawaban berupa blok Mermaid.")
    if mermaid_examples and flowchart_td_examples < mermaid_examples:
        warnings.append("Sebagian blok Mermaid tidak mendeklarasikan flowchart TD.")
    if max(assistant_lengths) > 12_000:
        warnings.append("Ada jawaban lebih dari 12.000 karakter; periksa risiko truncation.")

    return {
        "status": "valid_with_warnings" if warnings else "valid",
        "examples": total,
        "unique_examples": total - duplicate_examples,
        "duplicate_examples": duplicate_examples,
        "duplicate_ratio": round(duplicate_ratio, 6),
        "mermaid_examples": mermaid_examples,
        "mermaid_ratio": round(mermaid_ratio, 6),
        "flowchart_td_examples": flowchart_td_examples,
        "start_end_examples": start_end_examples,
        "decision_examples": decision_examples,
        "assistant_chars_min": min(assistant_lengths),
        "assistant_chars_max": max(assistant_lengths),
        "assistant_chars_average": round(sum(assistant_lengths) / total, 2),
        "warnings": warnings,
    }


def write_canonical_jsonl(records, path):
    path = Path(path)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def dataset_metadata_path(dataset_path):
    return Path(str(dataset_path) + ".meta.json")


def register_dataset(dataset_id, metadata):
    with dataset_registry_lock:
        dataset_registry[dataset_id] = dict(metadata)


def load_persisted_dataset_registry():
    with dataset_registry_lock:
        dataset_registry.clear()

        for meta_path in WEB_DATASET_DIR.glob("*.meta.json"):
            try:
                metadata = json.loads(meta_path.read_text(encoding="utf-8"))
                dataset_id = metadata["dataset_id"]
                dataset_path = Path(metadata["path"])

                if dataset_path.exists():
                    dataset_registry[dataset_id] = metadata
            except Exception:
                continue


def save_uploaded_dataset(file_storage, original_filename):
    safe_stem = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        Path(original_filename).stem,
    ).strip("._-") or "dataset"

    suffix = Path(original_filename).suffix.lower()
    if suffix not in {".json", ".jsonl"}:
        raise ValueError("File harus memiliki ekstensi .json atau .jsonl.")

    dataset_id = uuid.uuid4().hex
    temporary_path = WEB_DATASET_DIR / f".upload-{dataset_id}{suffix}"
    canonical_path = WEB_DATASET_DIR / f"{dataset_id}-{safe_stem}.jsonl"

    file_storage.save(str(temporary_path))

    try:
        if temporary_path.stat().st_size > MAX_UPLOAD_BYTES:
            raise ValueError("Ukuran file melebihi batas 100 MB.")

        records = load_records_from_file(temporary_path)
        validation_report = build_dataset_validation_report(records)
        write_canonical_jsonl(records, canonical_path)

        metadata = {
            "dataset_id": dataset_id,
            "name": original_filename,
            "canonical_name": canonical_path.name,
            "path": str(canonical_path),
            "examples": len(records),
            "size_bytes": canonical_path.stat().st_size,
            "uploaded_at": time.time(),
            "preview": records[:3],
            "validation_report": validation_report,
        }

        meta_path = dataset_metadata_path(canonical_path)
        meta_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        register_dataset(dataset_id, metadata)
        return metadata

    finally:
        temporary_path.unlink(missing_ok=True)


def get_dataset_metadata(dataset_id):
    with dataset_registry_lock:
        metadata = dataset_registry.get(dataset_id)
        return dict(metadata) if metadata else None


def list_dataset_metadata():
    with dataset_registry_lock:
        values = [dict(item) for item in dataset_registry.values()]

    return sorted(values, key=lambda item: item.get("uploaded_at", 0), reverse=True)


def load_canonical_dataset(dataset_id):
    metadata = get_dataset_metadata(dataset_id)
    if metadata is None:
        raise FileNotFoundError("Dataset tidak ditemukan.")

    path = Path(metadata["path"])
    if not path.exists():
        raise FileNotFoundError("File dataset tidak lagi tersedia.")

    return load_records_from_file(path), metadata


load_persisted_dataset_registry()


# ------------------------------------------------------------------
# Training preparation
# ------------------------------------------------------------------

def tokenize_records(records, max_seq_length):
    raw_dataset = Dataset.from_list(records)

    def tokenize_example(example):
        # Records sudah dinormalisasi saat upload/load. Dataset.map dapat
        # memberikan LazyRow yang bukan dict/Mapping pada versi datasets tertentu.
        try:
            messages = example["messages"]
        except (KeyError, TypeError) as exc:
            raise ValueError("Contoh hasil Dataset.map tidak memiliki field messages.") from exc
        prompt_messages = messages[:-1]

        full_text = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        prompt_text = qwen_tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        full_encoding = qwen_tokenizer(
            full_text,
            truncation=True,
            max_length=max_seq_length,
            add_special_tokens=False,
        )
        prompt_encoding = qwen_tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_seq_length,
            add_special_tokens=False,
        )

        input_ids = full_encoding["input_ids"]
        labels = input_ids.copy()
        prompt_length = min(len(prompt_encoding["input_ids"]), len(labels))
        labels[:prompt_length] = [-100] * prompt_length

        return {
            "input_ids": input_ids,
            "attention_mask": full_encoding["attention_mask"],
            "labels": labels,
            "length": len(input_ids),
        }

    tokenized = raw_dataset.map(
        tokenize_example,
        remove_columns=raw_dataset.column_names,
        desc="Tokenisasi dataset web",
    )

    tokenized = tokenized.filter(
        lambda example: any(label != -100 for label in example["labels"]),
        desc="Membuang contoh tanpa target",
    )

    if len(tokenized) == 0:
        raise RuntimeError(
            "Tidak ada contoh valid setelah tokenisasi. "
            "Naikkan max sequence length atau perbaiki dataset."
        )

    lengths = list(tokenized["length"])
    tokenized = tokenized.remove_columns(["length"])

    stats = {
        "examples": len(tokenized),
        "min_tokens": min(lengths),
        "max_tokens": max(lengths),
        "average_tokens": round(sum(lengths) / len(lengths), 2),
        "truncated_examples": sum(
            1 for length in lengths if length >= max_seq_length
        ),
    }
    return tokenized, stats


def cleanup_current_model():
    global qwen_model

    try:
        del qwen_model
    except Exception:
        pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_fresh_base_model(use_4bit):
    global qwen_model, USE_4BIT, COMPUTE_DTYPE

    if use_4bit and not torch.cuda.is_available():
        raise RuntimeError("QLoRA 4-bit memerlukan GPU CUDA.")

    cleanup_current_model()

    if torch.cuda.is_available():
        compute_dtype = (
            torch.bfloat16
            if torch.cuda.is_bf16_supported()
            else torch.float16
        )
    else:
        compute_dtype = torch.float32

    model_kwargs = {
        "torch_dtype": compute_dtype,
        "low_cpu_mem_usage": True,
        "trust_remote_code": False,
    }

    if use_4bit:
        model_kwargs["device_map"] = {"": 0}
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )

    qwen_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        **model_kwargs,
    )

    if torch.cuda.is_available() and not use_4bit:
        qwen_model.to("cuda")

    qwen_model.config.use_cache = False
    USE_4BIT = bool(use_4bit)
    COMPUTE_DTYPE = compute_dtype

    return qwen_model


def attach_training_parameters(model, mode, lora_r, lora_alpha, lora_dropout):
    mode = mode.lower()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    if mode in {"lora", "qlora"}:
        if mode == "qlora":
            model = prepare_model_for_kbit_training(
                model,
                use_gradient_checkpointing=True,
            )

        target_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ]

        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            target_modules=target_modules,
        )

        model = get_peft_model(model, lora_config)
        return model

    if mode == "full":
        for parameter in model.parameters():
            parameter.requires_grad = True
        return model

    raise ValueError("Mode fine-tuning tidak dikenali.")


def count_trainable_parameters(model):
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    total = sum(parameter.numel() for parameter in model.parameters())
    return {
        "trainable": trainable,
        "total": total,
        "percentage": round(100 * trainable / max(total, 1), 6),
    }


# ------------------------------------------------------------------
# Training jobs
# ------------------------------------------------------------------

def update_training_job(job_id, **values):
    with training_jobs_lock:
        job = training_jobs.get(job_id)
        if job is None:
            return

        job.update(values)
        job["updated_at"] = time.time()


def append_training_log(job_id, text):
    if not text:
        return

    with training_jobs_lock:
        job = training_jobs.get(job_id)
        if job is None:
            return

        logs = job.setdefault("logs", [])
        logs.append(str(text))
        if len(logs) > MAX_LOG_LINES:
            del logs[:-MAX_LOG_LINES]
        job["updated_at"] = time.time()


def get_training_job(job_id):
    with training_jobs_lock:
        job = training_jobs.get(job_id)
        return dict(job) if job else None


def training_is_running():
    with training_jobs_lock:
        return any(
            job.get("status") in {"queued", "preparing", "training", "saving"}
            for job in training_jobs.values()
        )


def training_cancel_requested(job_id):
    job = get_training_job(job_id)
    return bool(job and job.get("cancel_requested"))


def finish_cancelled_job(job_id, message="Fine-tuning dibatalkan."):
    update_training_job(
        job_id,
        status="cancelled",
        progress=100,
        message=message,
    )
    append_training_log(job_id, message)


class WebTrainingCallback(TrainerCallback):
    def __init__(self, job_id):
        self.job_id = job_id

    def _cancel_requested(self):
        job = get_training_job(self.job_id)
        return bool(job and job.get("cancel_requested"))

    def on_train_begin(self, args, state, control, **kwargs):
        update_training_job(
            self.job_id,
            status="training",
            progress=20,
            current_step=state.global_step,
            total_steps=state.max_steps,
            message="Fine-tuning dimulai.",
        )
        append_training_log(
            self.job_id,
            f"Training dimulai; total langkah: {state.max_steps}.",
        )

    def on_step_end(self, args, state, control, **kwargs):
        total_steps = max(int(state.max_steps or 0), 1)
        fraction = min(max(state.global_step / total_steps, 0.0), 1.0)
        progress = 20 + int(fraction * 72)

        update_training_job(
            self.job_id,
            progress=progress,
            current_step=state.global_step,
            total_steps=state.max_steps,
            epoch=state.epoch,
            message=f"Training langkah {state.global_step}/{state.max_steps}.",
        )

        if self._cancel_requested():
            append_training_log(
                self.job_id,
                "Permintaan pembatalan diterima; training dihentikan pada batas langkah.",
            )
            control.should_training_stop = True

        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        clean_logs = {}

        for key, value in logs.items():
            if isinstance(value, (int, float)):
                clean_logs[key] = round(float(value), 8)
            else:
                clean_logs[key] = value

        update_training_job(
            self.job_id,
            latest_metrics=clean_logs,
        )
        append_training_log(
            self.job_id,
            "step="
            + str(state.global_step)
            + " "
            + json.dumps(clean_logs, ensure_ascii=False),
        )

    def on_save(self, args, state, control, **kwargs):
        append_training_log(
            self.job_id,
            f"Checkpoint disimpan pada langkah {state.global_step}.",
        )

    def on_train_end(self, args, state, control, **kwargs):
        append_training_log(
            self.job_id,
            f"Trainer berhenti pada langkah {state.global_step}.",
        )


def sanitize_adapter_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "-", str(name or "")).strip(".-")
    return cleaned[:80] or f"mermaid-adapter-{time.strftime('%Y%m%d-%H%M%S')}"


def run_training_job(job_id, config):
    global qwen_model, MODEL_RUNTIME_STATE, active_training_job_id

    acquired = False
    trainer = None

    try:
        with active_training_lock:
            if active_training_job_id not in (None, job_id):
                raise RuntimeError("Ada fine-tuning lain yang masih aktif.")
            active_training_job_id = job_id

        update_training_job(
            job_id,
            status="preparing",
            progress=3,
            message="Memuat dan memvalidasi dataset.",
        )

        records, dataset_meta = load_canonical_dataset(config["dataset_id"])
        append_training_log(
            job_id,
            f"Dataset: {dataset_meta['name']} ({len(records):,} contoh). Backend patch {NOTEBOOK_PATCH_VERSION}.",
        )

        if training_cancel_requested(job_id):
            finish_cancelled_job(job_id)
            return

        model_operation_lock.acquire()
        acquired = True

        mode = config["mode"]
        use_4bit = mode == "qlora"

        free_disk_bytes = shutil.disk_usage(WEB_WORK_DIR).free
        minimum_free_bytes = (
            10 * 1024**3
            if mode == "full"
            else 1 * 1024**3
        )
        update_training_job(
            job_id,
            free_disk_bytes=free_disk_bytes,
        )

        if free_disk_bytes < minimum_free_bytes:
            raise RuntimeError(
                "Ruang disk tidak mencukupi. "
                f"Mode {mode} membutuhkan setidaknya "
                f"{minimum_free_bytes / 1024**3:.1f} GB ruang kosong; "
                f"tersedia {free_disk_bytes / 1024**3:.1f} GB."
            )

        update_training_job(
            job_id,
            progress=7,
            message="Memuat ulang model dasar.",
        )
        append_training_log(
            job_id,
            f"Memuat {MODEL_ID}; mode={mode}; 4-bit={use_4bit}.",
        )

        qwen_model = load_fresh_base_model(use_4bit=use_4bit)

        if training_cancel_requested(job_id):
            qwen_model.eval()
            qwen_model.config.use_cache = True
            MODEL_RUNTIME_STATE = {
                "model_id": MODEL_ID,
                "mode": "base-4bit" if use_4bit else "base",
                "adapter_name": None,
                "output_dir": None,
                "updated_at": time.time(),
            }
            finish_cancelled_job(job_id)
            return

        update_training_job(
            job_id,
            progress=11,
            message="Melakukan tokenisasi dataset.",
        )

        validation_ratio = config["validation_ratio"]
        if validation_ratio > 0 and len(records) < 2:
            raise ValueError("Validation split memerlukan minimal dua contoh.")

        if validation_ratio > 0:
            raw_dataset = Dataset.from_list(records).train_test_split(
                test_size=validation_ratio,
                seed=config["seed"],
                shuffle=True,
            )
            train_records = raw_dataset["train"].to_list()
            validation_records = raw_dataset["test"].to_list()
        else:
            train_records = records
            validation_records = []

        tokenized_dataset, token_stats = tokenize_records(
            train_records,
            max_seq_length=config["max_seq_length"],
        )
        validation_dataset = None
        validation_token_stats = None
        if validation_records:
            validation_dataset, validation_token_stats = tokenize_records(
                validation_records,
                max_seq_length=config["max_seq_length"],
            )

        token_stats["train_examples"] = len(tokenized_dataset)
        token_stats["validation_examples"] = len(validation_dataset or [])
        update_training_job(
            job_id,
            token_stats=token_stats,
            validation_token_stats=validation_token_stats,
        )
        append_training_log(
            job_id,
            "Statistik token: " + json.dumps(token_stats, ensure_ascii=False),
        )

        if training_cancel_requested(job_id):
            qwen_model.eval()
            qwen_model.config.use_cache = True
            MODEL_RUNTIME_STATE = {
                "model_id": MODEL_ID,
                "mode": "base-4bit" if use_4bit else "base",
                "adapter_name": None,
                "output_dir": None,
                "updated_at": time.time(),
            }
            finish_cancelled_job(job_id)
            return

        update_training_job(
            job_id,
            progress=15,
            message="Memasang parameter fine-tuning.",
        )

        qwen_model = attach_training_parameters(
            qwen_model,
            mode=mode,
            lora_r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
        )

        if training_cancel_requested(job_id):
            append_training_log(
                job_id,
                "Pembatalan diterima sebelum Trainer dimulai; model dasar dipulihkan.",
            )
            qwen_model = load_fresh_base_model(use_4bit=use_4bit)
            qwen_model.eval()
            qwen_model.config.use_cache = True
            MODEL_RUNTIME_STATE = {
                "model_id": MODEL_ID,
                "mode": "base-4bit" if use_4bit else "base",
                "adapter_name": None,
                "output_dir": None,
                "updated_at": time.time(),
            }
            finish_cancelled_job(job_id)
            return

        parameter_stats = count_trainable_parameters(qwen_model)
        update_training_job(job_id, parameter_stats=parameter_stats)
        append_training_log(
            job_id,
            "Parameter: " + json.dumps(parameter_stats, ensure_ascii=False),
        )

        adapter_name = sanitize_adapter_name(config["adapter_name"])
        output_dir = WEB_OUTPUT_DIR / f"{job_id}-{adapter_name}"
        output_dir.mkdir(parents=True, exist_ok=False)

        use_bf16 = (
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        )
        use_fp16 = torch.cuda.is_available() and not use_bf16

        data_collator = DataCollatorForSeq2Seq(
            tokenizer=qwen_tokenizer,
            model=qwen_model,
            padding=True,
            label_pad_token_id=-100,
            return_tensors="pt",
        )

        training_args = TrainingArguments(
            output_dir=str(output_dir),
            overwrite_output_dir=False,
            num_train_epochs=config["epochs"],
            max_steps=config["max_steps"],
            per_device_train_batch_size=config["batch_size"],
            gradient_accumulation_steps=config["gradient_accumulation_steps"],
            learning_rate=config["learning_rate"],
            warmup_ratio=config["warmup_ratio"],
            lr_scheduler_type="cosine",
            weight_decay=config["weight_decay"],
            max_grad_norm=1.0,
            logging_steps=config["logging_steps"],
            logging_first_step=True,
            save_strategy="steps",
            save_steps=config["save_steps"],
            eval_strategy="steps" if validation_dataset is not None else "no",
            eval_steps=config["save_steps"],
            load_best_model_at_end=validation_dataset is not None,
            metric_for_best_model="eval_loss" if validation_dataset is not None else None,
            greater_is_better=False if validation_dataset is not None else None,
            save_total_limit=2,
            bf16=use_bf16,
            fp16=use_fp16,
            gradient_checkpointing=True,
            optim="paged_adamw_8bit" if mode == "qlora" else "adamw_torch",
            report_to="none",
            remove_unused_columns=False,
            dataloader_pin_memory=torch.cuda.is_available(),
            group_by_length=True,
            seed=config["seed"],
            save_safetensors=True,
        )

        trainer_kwargs = {
            "model": qwen_model,
            "args": training_args,
            "train_dataset": tokenized_dataset,
            "eval_dataset": validation_dataset,
            "data_collator": data_collator,
            "callbacks": [WebTrainingCallback(job_id)],
        }

        trainer_parameters = inspect.signature(Trainer.__init__).parameters
        if "processing_class" in trainer_parameters:
            trainer_kwargs["processing_class"] = qwen_tokenizer
        else:
            trainer_kwargs["tokenizer"] = qwen_tokenizer

        trainer = Trainer(**trainer_kwargs)
        train_result = trainer.train()

        current_job = get_training_job(job_id)
        if current_job and current_job.get("cancel_requested"):
            append_training_log(
                job_id,
                "Memulihkan model dasar agar hasil parsial tidak menjadi model aktif.",
            )

            try:
                trainer.model = None
            except Exception:
                pass
            del train_result
            del trainer
            trainer = None
            shutil.rmtree(output_dir, ignore_errors=True)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            qwen_model = load_fresh_base_model(use_4bit=use_4bit)
            qwen_model.eval()
            qwen_model.config.use_cache = True
            MODEL_RUNTIME_STATE = {
                "model_id": MODEL_ID,
                "mode": "base-4bit" if use_4bit else "base",
                "adapter_name": None,
                "output_dir": None,
                "updated_at": time.time(),
            }
            finish_cancelled_job(
                job_id,
                "Fine-tuning dibatalkan. Model dasar telah dipulihkan.",
            )
            return

        update_training_job(
            job_id,
            status="saving",
            progress=94,
            message="Menyimpan model/adapter.",
        )

        trainer.save_model(str(output_dir))
        qwen_tokenizer.save_pretrained(str(output_dir))

        metrics = {}
        for key, value in dict(train_result.metrics).items():
            if isinstance(value, (int, float)):
                metrics[key] = round(float(value), 8)
            else:
                metrics[key] = value

        summary = {
            "job_id": job_id,
            "dataset": dataset_meta,
            "configuration": config,
            "token_stats": token_stats,
            "parameter_stats": parameter_stats,
            "metrics": metrics,
            "model_id": MODEL_ID,
            "completed_at": time.time(),
        }

        (output_dir / "training_summary.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        qwen_model.eval()
        qwen_model.config.use_cache = True

        MODEL_RUNTIME_STATE = {
            "model_id": MODEL_ID,
            "mode": mode,
            "adapter_name": adapter_name,
            "output_dir": str(output_dir),
            "updated_at": time.time(),
        }

        update_training_job(
            job_id,
            status="done",
            progress=100,
            message="Fine-tuning selesai; hasil menjadi model aktif.",
            output_dir=str(output_dir),
            adapter_name=adapter_name,
            metrics=metrics,
            downloadable=True,
            model_runtime_state=dict(MODEL_RUNTIME_STATE),
        )
        append_training_log(job_id, "Fine-tuning dan penyimpanan selesai.")

    except Exception as exc:
        traceback_text = traceback.format_exc()
        append_training_log(job_id, traceback_text)
        update_training_job(
            job_id,
            status="error",
            progress=100,
            message=f"Fine-tuning gagal: {exc}",
            error=str(exc),
        )

        try:
            if "qwen_model" in globals():
                qwen_model.eval()
                qwen_model.config.use_cache = True
        except Exception:
            pass

    finally:
        if trainer is not None:
            del trainer

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if acquired:
            model_operation_lock.release()

        with active_training_lock:
            if active_training_job_id == job_id:
                active_training_job_id = None


def parse_training_config(payload):
    mode = str(payload.get("mode", "lora")).strip().lower()
    if mode not in {"lora", "qlora", "full"}:
        raise ValueError("Mode harus lora, qlora, atau full.")

    def integer_field(name, default, minimum, maximum):
        try:
            value = int(payload.get(name, default))
        except Exception as exc:
            raise ValueError(f"{name} harus berupa bilangan bulat.") from exc

        if value < minimum or value > maximum:
            raise ValueError(
                f"{name} harus berada pada rentang {minimum}–{maximum}."
            )
        return value

    def float_field(name, default, minimum, maximum):
        try:
            value = float(payload.get(name, default))
        except Exception as exc:
            raise ValueError(f"{name} harus berupa angka.") from exc

        if value < minimum or value > maximum:
            raise ValueError(
                f"{name} harus berada pada rentang {minimum}–{maximum}."
            )
        return value

    max_steps = integer_field("max_steps", -1, -1, 1_000_000)
    if max_steps == 0:
        raise ValueError("max_steps harus -1 atau minimal 1.")

    default_lr = 2e-5 if mode == "full" else 2e-4

    config = {
        "dataset_id": str(payload.get("dataset_id", "")).strip(),
        "adapter_name": sanitize_adapter_name(payload.get("adapter_name")),
        "mode": mode,
        "max_seq_length": integer_field(
            "max_seq_length", 1024, 128, 8192
        ),
        "epochs": float_field("epochs", 3.0, 0.01, 1000.0),
        "max_steps": max_steps,
        "batch_size": integer_field("batch_size", 1, 1, 64),
        "gradient_accumulation_steps": integer_field(
            "gradient_accumulation_steps", 8, 1, 1024
        ),
        "learning_rate": float_field(
            "learning_rate", default_lr, 1e-8, 1.0
        ),
        "warmup_ratio": float_field("warmup_ratio", 0.03, 0.0, 1.0),
        "weight_decay": float_field("weight_decay", 0.0, 0.0, 10.0),
        "logging_steps": integer_field("logging_steps", 1, 1, 100_000),
        "save_steps": integer_field("save_steps", 50, 1, 1_000_000),
        "lora_r": integer_field("lora_r", 16, 1, 1024),
        "lora_alpha": integer_field("lora_alpha", 32, 1, 65_536),
        "lora_dropout": float_field("lora_dropout", 0.05, 0.0, 0.99),
        "seed": integer_field("seed", 42, 0, 2_147_483_647),
        "validation_ratio": float_field("validation_ratio", 0.1, 0.0, 0.5),
    }

    if not config["dataset_id"]:
        raise ValueError("Pilih dataset terlebih dahulu.")

    if get_dataset_metadata(config["dataset_id"]) is None:
        raise ValueError("Dataset yang dipilih tidak ditemukan.")

    if mode == "qlora" and not torch.cuda.is_available():
        raise ValueError("QLoRA memerlukan GPU CUDA.")

    return config


In [ ]:
# 5) Aplikasi web: Mermaid + upload dataset + fine-tuning UI

import os
import re
import signal
import shutil
import subprocess
import threading
import time
import uuid
from pathlib import Path

import requests
from flask import (
    Flask,
    jsonify,
    render_template_string,
    request,
    send_file,
)
from werkzeug.serving import make_server
from werkzeug.utils import secure_filename


# ============================================================
# Configuration
# ============================================================

APP_PORT = 7860
MERMAID_VERSION = "11"
CLOUDFLARED_LOG_PATH = str(
    (Path("/content") if Path("/content").exists() else Path.cwd())
    / "qwen-mermaid-cloudflared.log"
)

DEFAULT_FLOWCHART_REQUEST = (
    "Buat flowchart Mermaid yang jelas untuk proses pemesanan daring. "
    "Sertakan pemilihan produk, checkout, validasi pembayaran, pemeriksaan "
    "stok, pengemasan, pengiriman, konfirmasi penerimaan, dan jalur kegagalan."
)

MERMAID_GENERATION_INSTRUCTIONS = """
You generate Mermaid version 11 flowchart code.
Return only Mermaid code, with no Markdown fences and no explanation.
Use flowchart TD or flowchart LR unless the user requests another direction.
Use short, readable node labels wrapped in quotes when labels contain spaces.
Use stable node ids made from letters, numbers, and underscores only.
Use Mermaid flowchart syntax supported by Mermaid 11.
Keep the diagram readable by grouping related steps with subgraph when useful.
Do not include raw HTML, JavaScript, external URLs, or styling that requires plugins.
""".strip()


# ============================================================
# Generation jobs
# ============================================================

generation_jobs = {}
generation_jobs_lock = threading.Lock()


def update_generation_job(job_id, **values):
    with generation_jobs_lock:
        if job_id in generation_jobs:
            generation_jobs[job_id].update(values)


def get_generation_job(job_id):
    with generation_jobs_lock:
        job = generation_jobs.get(job_id)
        return dict(job) if job else None


def extract_mermaid_code(text):
    text = (text or "").strip()

    fence_match = re.search(
        r"```(?:mermaid|mmd)?\s*([\s\S]*?)```",
        text,
        flags=re.IGNORECASE,
    )
    if fence_match:
        text = fence_match.group(1).strip()

    lines = text.splitlines()

    while lines and lines[0].strip().lower() in {"mermaid", "mmd"}:
        lines.pop(0)

    flowchart_start = re.compile(
        r"^\s*(flowchart|graph)\s+(TD|TB|BT|RL|LR)\b",
        flags=re.IGNORECASE,
    )

    start_index = next(
        (
            index
            for index, line in enumerate(lines)
            if flowchart_start.search(line)
        ),
        None,
    )

    if start_index is not None:
        lines = lines[start_index:]

    trailing_prose = re.compile(
        r"^\s*(explanation|note|notes|summary)\s*:",
        flags=re.IGNORECASE,
    )

    for index, line in enumerate(lines):
        if trailing_prose.search(line):
            lines = lines[:index]
            break

    return "\n".join(lines).strip()


def looks_like_flowchart(code):
    return bool(
        re.match(
            r"^\s*(flowchart|graph)\s+(TD|TB|BT|RL|LR)\b",
            code or "",
            flags=re.IGNORECASE,
        )
    )


def generate_mermaid_with_model(flowchart_request):
    if training_is_running():
        raise RuntimeError(
            "Fine-tuning sedang berlangsung. Tunggu sampai selesai atau batalkan training."
        )

    user_prompt = f"""{MERMAID_GENERATION_INSTRUCTIONS}

User flowchart request:
{flowchart_request.strip()}

Return only valid Mermaid v11 flowchart code."""

    acquired = model_operation_lock.acquire(timeout=3)
    if not acquired:
        raise RuntimeError("Model sedang dipakai oleh proses lain.")

    try:
        raw_text = generate_qwen_text(
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a precise Mermaid v11 flowchart generator. "
                        "Return only valid Mermaid code without Markdown fences."
                    ),
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            max_new_tokens=2048,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    finally:
        model_operation_lock.release()

    return raw_text, extract_mermaid_code(raw_text)


def process_generation_job(job_id, flowchart_request):
    try:
        update_generation_job(
            job_id,
            status="processing",
            progress=15,
            message="Mengirim prompt ke model.",
        )

        raw_response, mermaid_code = generate_mermaid_with_model(
            flowchart_request
        )

        warning = None
        if not looks_like_flowchart(mermaid_code):
            warning = (
                "Respons model telah dibersihkan, tetapi tidak diawali deklarasi "
                "flowchart Mermaid. Kode masih dapat diedit dan dirender ulang."
            )

        update_generation_job(
            job_id,
            status="done",
            progress=100,
            message="Kode Mermaid berhasil dibuat.",
            raw_response=raw_response,
            mermaid_code=mermaid_code,
            warning=warning,
        )

    except Exception as exc:
        update_generation_job(
            job_id,
            status="error",
            progress=100,
            message=f"Generasi gagal: {exc}",
            error=str(exc),
        )


# ============================================================
# Flask app
# ============================================================

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = MAX_UPLOAD_BYTES

PAGE = r"""
<!doctype html>
<html lang="id">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Qwen2.5 Mermaid & Fine-Tuning Studio</title>

  <style>
    :root {
      color-scheme: light dark;
      font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background: #f3f6fa;
      color: #172033;
    }

    * { box-sizing: border-box; }
    body { margin: 0; min-height: 100vh; background: #f3f6fa; color: #172033; }
    button, textarea, input, select { font: inherit; }

    button {
      border: 0;
      border-radius: 8px;
      padding: 10px 14px;
      background: #0f766e;
      color: #fff;
      font-weight: 700;
      cursor: pointer;
      min-height: 42px;
    }

    button:hover { background: #0b5f59; }
    button:disabled { background: #94a3b8; cursor: not-allowed; }

    .secondary {
      border: 1px solid #b8c3d4;
      background: #fff;
      color: #172033;
    }
    .secondary:hover { background: #edf2f7; }

    .danger { background: #b91c1c; }
    .danger:hover { background: #991b1b; }

    .shell {
      width: min(1500px, calc(100vw - 28px));
      margin: 0 auto;
      padding: 18px 0 30px;
    }

    .topbar {
      display: flex;
      justify-content: space-between;
      align-items: flex-start;
      gap: 18px;
      margin-bottom: 14px;
    }

    h1, h2, h3, p { margin-top: 0; }
    h1 { margin-bottom: 7px; font-size: clamp(24px, 3vw, 36px); }
    h2 { margin-bottom: 0; font-size: 17px; }
    h3 { font-size: 14px; margin-bottom: 8px; }

    .subtitle { margin: 0; color: #64748b; line-height: 1.5; }

    .badges { display: flex; flex-wrap: wrap; gap: 8px; justify-content: flex-end; }

    .badge {
      display: inline-flex;
      align-items: center;
      min-height: 31px;
      border: 1px solid #cbd5e1;
      border-radius: 999px;
      padding: 5px 11px;
      background: #fff;
      color: #475569;
      font-size: 13px;
      font-weight: 700;
      white-space: nowrap;
    }

    .badge[data-state="busy"] {
      border-color: #f59e0b;
      background: #fff7e6;
      color: #8a4b00;
    }
    .badge[data-state="ok"] {
      border-color: #14b8a6;
      background: #e7fbf8;
      color: #0f766e;
    }
    .badge[data-state="error"] {
      border-color: #ef4444;
      background: #fff0f0;
      color: #b91c1c;
    }

    .tabs {
      display: flex;
      gap: 8px;
      padding: 7px;
      border: 1px solid #d8e0ea;
      border-radius: 10px;
      background: #fff;
      margin-bottom: 14px;
    }

    .tab {
      background: transparent;
      color: #475569;
      min-height: 38px;
      padding: 8px 14px;
    }
    .tab:hover { background: #eef3f8; }
    .tab.active { background: #0f766e; color: #fff; }

    .tab-page { display: none; }
    .tab-page.active { display: block; }

    .grid {
      display: grid;
      gap: 14px;
      align-items: stretch;
    }
    .diagram-grid {
      grid-template-columns: minmax(280px, .8fr) minmax(320px, 1fr) minmax(420px, 1.35fr);
    }
    .training-grid {
      grid-template-columns: minmax(320px, .85fr) minmax(360px, 1fr) minmax(360px, 1fr);
    }

    .panel, .status-band {
      border: 1px solid #d8e0ea;
      border-radius: 10px;
      background: #fff;
      overflow: hidden;
    }

    .panel {
      min-height: 530px;
      display: flex;
      flex-direction: column;
    }

    .panel-head {
      min-height: 56px;
      padding: 12px 14px;
      border-bottom: 1px solid #d8e0ea;
      background: #fbfcfe;
      display: flex;
      justify-content: space-between;
      align-items: center;
      gap: 10px;
    }

    .panel-body { padding: 14px; flex: 1; min-height: 0; }
    .stack { display: grid; gap: 12px; }
    .row { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; }
    .row > * { flex: 1; }

    label {
      display: grid;
      gap: 6px;
      font-size: 13px;
      font-weight: 700;
      color: #475569;
    }

    input, select, textarea {
      width: 100%;
      border: 1px solid #c7d1df;
      border-radius: 8px;
      padding: 10px 11px;
      background: #fbfcfe;
      color: #172033;
    }

    textarea { resize: vertical; line-height: 1.45; }
    #promptBox { min-height: 360px; }
    #codeEditor { min-height: 420px; font-family: "JetBrains Mono", Consolas, monospace; font-size: 13px; }
    #trainingLog { min-height: 260px; font-family: "JetBrains Mono", Consolas, monospace; font-size: 12px; }

    .actions {
      display: flex;
      gap: 9px;
      flex-wrap: wrap;
      justify-content: flex-end;
      margin-top: 12px;
    }

    .preview-wrap { padding: 14px; background: #f8fafc; flex: 1; display: flex; flex-direction: column; }
    .diagram {
      flex: 1;
      min-height: 420px;
      border: 1px solid #cfd8e6;
      border-radius: 8px;
      background: #fff;
      display: grid;
      place-items: center;
      overflow: auto;
      padding: 18px;
    }
    .diagram svg { max-width: 100%; height: auto; }

    .placeholder { color: #64748b; text-align: center; max-width: 38ch; line-height: 1.45; }

    .status-band { margin-top: 14px; padding: 12px 14px; }
    .progress-track {
      height: 12px;
      border: 1px solid #ccd6e3;
      border-radius: 999px;
      background: #e2e8f0;
      overflow: hidden;
      margin-bottom: 9px;
    }
    .progress-bar { width: 0%; height: 100%; background: #1d4ed8; transition: width .25s ease; }
    .status-text {
      margin: 0;
      white-space: pre-wrap;
      font: 12px/1.45 "JetBrains Mono", Consolas, monospace;
      color: #475569;
    }

    .notice, .error, .success {
      border-radius: 8px;
      padding: 10px 11px;
      line-height: 1.45;
      white-space: pre-wrap;
      font-size: 13px;
    }
    .notice { border: 1px solid #cbd5e1; background: #f8fafc; color: #475569; }
    .error { border: 1px solid #fca5a5; background: #fff1f2; color: #991b1b; }
    .success { border: 1px solid #5eead4; background: #ecfdf5; color: #0f766e; }
    [hidden] { display: none !important; }

    .dataset-list { display: grid; gap: 8px; margin-top: 10px; }
    .dataset-item {
      border: 1px solid #d8e0ea;
      border-radius: 8px;
      padding: 10px;
      background: #fbfcfe;
    }
    .dataset-title { font-weight: 800; overflow-wrap: anywhere; }
    .muted { color: #64748b; font-size: 12px; line-height: 1.45; }
    .preview-json {
      max-height: 220px;
      overflow: auto;
      white-space: pre-wrap;
      word-break: break-word;
      font: 11px/1.4 "JetBrains Mono", Consolas, monospace;
      border: 1px solid #d8e0ea;
      border-radius: 8px;
      padding: 9px;
      background: #f8fafc;
    }

    details {
      border: 1px solid #d8e0ea;
      border-radius: 8px;
      padding: 10px;
      background: #fbfcfe;
    }
    summary { cursor: pointer; font-weight: 800; }

    @media (max-width: 1200px) {
      .diagram-grid, .training-grid { grid-template-columns: 1fr 1fr; }
      .diagram-grid .preview-panel, .training-grid .log-panel { grid-column: 1 / -1; }
    }

    @media (max-width: 760px) {
      .shell { width: min(100% - 18px, 720px); padding-top: 10px; }
      .topbar, .diagram-grid, .training-grid { display: grid; grid-template-columns: 1fr; }
      .badges { justify-content: flex-start; }
      .tabs { overflow-x: auto; }
      .panel { min-height: 420px; }
      .training-grid .log-panel { grid-column: auto; }
      .row { display: grid; grid-template-columns: 1fr; }
      button { width: 100%; }
    }

    @media (prefers-color-scheme: dark) {
      :root, body { background: #10151d; color: #f7fafc; }
      .panel, .status-band, .tabs, .badge { background: #171e29; border-color: #334155; }
      .panel-head { background: #1c2532; border-color: #334155; }
      input, select, textarea, .diagram, .dataset-item, .preview-json, details {
        background: #111827; border-color: #405169; color: #f7fafc;
      }
      .preview-wrap { background: #151c27; }
      .secondary { background: #1f2937; border-color: #475569; color: #f7fafc; }
      .secondary:hover { background: #293548; }
      .subtitle, label, .muted, .status-text, .placeholder, .badge { color: #cbd5e1; }
      .notice { background: #1f2937; border-color: #475569; color: #cbd5e1; }
      .success { background: #073b35; border-color: #0f766e; color: #99f6e4; }
      .error { background: #3b161a; border-color: #7f1d1d; color: #fecdd3; }
      .progress-track { background: #253043; border-color: #405169; }
    }
  </style>
</head>
<body>
<div class="shell">
  <header class="topbar">
    <div>
      <h1>Qwen2.5 Mermaid & Fine-Tuning Studio</h1>
      <p class="subtitle">Generator flowchart dan pelatihan model Qwen2.5-1.5B dari browser.</p>
    </div>
    <div class="badges">
      <span class="badge" id="modelBadge">Model: memuat status…</span>
      <span class="badge" id="gpuBadge">GPU: memeriksa…</span>
      <span class="badge" id="runtimeBadge" data-state="ok">Siap</span>
    </div>
  </header>

  <nav class="tabs" aria-label="Fitur aplikasi">
    <button class="tab active" data-tab="diagramPage" type="button">Generator Mermaid</button>
    <button class="tab" data-tab="trainingPage" type="button">Dataset & Fine-Tuning</button>
  </nav>

  <section id="diagramPage" class="tab-page active">
    <main class="grid diagram-grid">
      <section class="panel">
        <div class="panel-head"><h2>Prompt</h2></div>
        <div class="panel-body stack">
          <textarea id="promptBox" spellcheck="true">{{ default_prompt }}</textarea>
          <div class="actions">
            <button type="button" id="generateButton">Buat Diagram</button>
          </div>
        </div>
      </section>

      <section class="panel">
        <div class="panel-head">
          <h2>Kode Mermaid</h2>
          <button class="secondary" type="button" id="renderButton">Render</button>
        </div>
        <div class="panel-body">
          <textarea id="codeEditor" spellcheck="false"></textarea>
        </div>
      </section>

      <section class="panel preview-panel">
        <div class="panel-head">
          <h2>Preview</h2>
          <div class="row" style="flex:0 0 auto">
            <button class="secondary" type="button" id="downloadSvgButton">SVG</button>
            <button class="secondary" type="button" id="downloadPngButton">PNG</button>
          </div>
        </div>
        <div class="preview-wrap">
          <div class="diagram" id="diagramBox">
            <div class="placeholder">Preview flowchart akan muncul di sini.</div>
          </div>
          <div class="error" id="renderError" hidden></div>
        </div>
      </section>
    </main>

    <section class="status-band">
      <div class="progress-track"><div class="progress-bar" id="generationProgress"></div></div>
      <pre class="status-text" id="generationStatus">Status: siap.</pre>
    </section>
  </section>

  <section id="trainingPage" class="tab-page">
    <main class="grid training-grid">
      <section class="panel">
        <div class="panel-head"><h2>1. Unggah Dataset</h2></div>
        <div class="panel-body stack">
          <div class="notice">Format: .jsonl atau .json. Diterima: messages, prompt-completion, atau instruction-output. Maksimum 100 MB.</div>

          <label>
            Berkas dataset
            <input id="datasetFile" type="file" accept=".json,.jsonl,application/json">
          </label>

          <div class="actions">
            <button type="button" id="uploadDatasetButton">Unggah & Validasi</button>
            <button class="secondary" type="button" id="refreshDatasetsButton">Muat Ulang Daftar</button>
          </div>

          <div id="uploadMessage" class="notice">Belum ada unggahan pada sesi ini.</div>

          <div>
            <h3>Dataset tersedia</h3>
            <div class="dataset-list" id="datasetList">
              <div class="placeholder">Belum ada dataset.</div>
            </div>
          </div>

          <div>
            <h3>Preview dataset</h3>
            <pre class="preview-json" id="datasetPreview">Pilih atau unggah dataset.</pre>
          </div>
        </div>
      </section>

      <section class="panel">
        <div class="panel-head"><h2>2. Konfigurasi Fine-Tuning</h2></div>
        <div class="panel-body stack">
          <label>
            Dataset
            <select id="datasetSelect"><option value="">Pilih dataset</option></select>
          </label>

          <div class="row">
            <label>
              Mode
              <select id="trainingMode">
                <option value="lora">LoRA (FP16/BF16)</option>
                <option value="qlora">QLoRA 4-bit</option>
                <option value="full">Full fine-tuning</option>
              </select>
            </label>
            <label>
              Nama adapter/model
              <input id="adapterName" value="mermaid-qwen-adapter">
            </label>
          </div>

          <div class="row">
            <label>
              Epoch
              <input id="epochs" type="number" min="0.01" step="0.1" value="3">
            </label>
            <label>
              Max steps
              <input id="maxSteps" type="number" min="-1" step="1" value="-1">
            </label>
            <label>
              Max sequence
              <select id="maxSeqLength">
                <option value="512">512</option>
                <option value="1024" selected>1024</option>
                <option value="2048">2048</option>
                <option value="4096">4096</option>
              </select>
            </label>
          </div>

          <div class="row">
            <label>
              Batch/device
              <input id="batchSize" type="number" min="1" value="1">
            </label>
            <label>
              Gradient accumulation
              <input id="gradientAccumulation" type="number" min="1" value="8">
            </label>
            <label>
              Learning rate
              <input id="learningRate" type="number" min="0.00000001" step="0.00001" value="0.0002">
            </label>
          </div>

          <details>
            <summary>Pengaturan lanjutan</summary>
            <div class="stack" style="margin-top:12px">
              <div class="row">
                <label>LoRA r<input id="loraR" type="number" min="1" value="16"></label>
                <label>LoRA alpha<input id="loraAlpha" type="number" min="1" value="32"></label>
                <label>LoRA dropout<input id="loraDropout" type="number" min="0" max="0.99" step="0.01" value="0.05"></label>
              </div>
              <div class="row">
                <label>Warmup ratio<input id="warmupRatio" type="number" min="0" max="1" step="0.01" value="0.03"></label>
                <label>Weight decay<input id="weightDecay" type="number" min="0" step="0.01" value="0"></label>
                <label>Seed<input id="seed" type="number" min="0" value="42"></label>
                <label>Validation split<input id="validationRatio" type="number" min="0" max="0.5" step="0.05" value="0.1"></label>
              </div>
              <div class="row">
                <label>Logging steps<input id="loggingSteps" type="number" min="1" value="1"></label>
                <label>Save steps<input id="saveSteps" type="number" min="1" value="50"></label>
              </div>
            </div>
          </details>

          <div class="notice" id="modeNotice">
            LoRA memuat ulang model dasar dalam FP16/BF16 dan melatih adapter.
          </div>

          <div class="actions">
            <button type="button" id="startTrainingButton">Mulai Fine-Tuning</button>
            <button class="danger" type="button" id="cancelTrainingButton" disabled>Batalkan</button>
          </div>
        </div>
      </section>

      <section class="panel log-panel">
        <div class="panel-head">
          <h2>3. Progres & Hasil</h2>
          <button class="secondary" id="downloadAdapterButton" type="button" disabled>Unduh Hasil ZIP</button>
        </div>
        <div class="panel-body stack">
          <div class="progress-track"><div class="progress-bar" id="trainingProgress"></div></div>
          <pre class="status-text" id="trainingStatus">Belum ada training.</pre>
          <textarea id="trainingLog" readonly spellcheck="false"></textarea>
          <div class="success" id="trainingSuccess" hidden></div>
          <div class="error" id="trainingError" hidden></div>
        </div>
      </section>
    </main>
  </section>
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@{{ mermaid_version }}/dist/mermaid.esm.min.mjs";

mermaid.initialize({
  startOnLoad: false,
  securityLevel: "strict",
  theme: "base",
  flowchart: { htmlLabels: false, curve: "basis" },
  themeVariables: {
    primaryColor: "#dff7f3",
    primaryTextColor: "#172033",
    primaryBorderColor: "#0f766e",
    lineColor: "#1d4ed8",
    secondaryColor: "#fff7e6",
    tertiaryColor: "#f8fafc"
  }
});

const $ = (id) => document.getElementById(id);
const modelBadge = $("modelBadge");
const gpuBadge = $("gpuBadge");
const runtimeBadge = $("runtimeBadge");

let generationPoll = null;
let trainingPoll = null;
let activeTrainingJobId = null;
let renderCounter = 0;
let currentDatasets = [];

function setBar(element, percent) {
  const safe = Math.max(0, Math.min(100, Number(percent) || 0));
  element.style.width = safe + "%";
}

function setRuntime(text, state="ok") {
  runtimeBadge.textContent = text;
  runtimeBadge.dataset.state = state;
}

function showBox(element, message) {
  element.hidden = !message;
  element.textContent = message || "";
}

document.querySelectorAll(".tab").forEach((button) => {
  button.addEventListener("click", () => {
    document.querySelectorAll(".tab").forEach((item) => item.classList.remove("active"));
    document.querySelectorAll(".tab-page").forEach((item) => item.classList.remove("active"));
    button.classList.add("active");
    $(button.dataset.tab).classList.add("active");
  });
});

async function fetchJson(url, options={}) {
  const response = await fetch(url, options);
  let payload;
  try {
    payload = await response.json();
  } catch {
    payload = { success: false, error: await response.text() };
  }
  if (!response.ok || payload.success === false) {
    throw new Error(payload.error || payload.message || "Permintaan gagal.");
  }
  return payload;
}

async function refreshSystemStatus() {
  try {
    const result = await fetchJson("/api/system", { cache: "no-store" });
    const state = result.model_runtime_state || {};
    const label = state.adapter_name
      ? `${state.mode}: ${state.adapter_name}`
      : (state.mode || "base");
    modelBadge.textContent = "Model: " + label;
    gpuBadge.textContent = result.gpu.available
      ? `GPU: ${result.gpu.name}`
      : "GPU: tidak tersedia";
    if (result.training_running) {
      setRuntime("Training", "busy");
    }
  } catch (error) {
    modelBadge.textContent = "Model: status gagal";
  }
}

async function renderMermaid(code) {
  const trimmed = (code || "").trim();
  if (!trimmed) {
    $("diagramBox").innerHTML = '<div class="placeholder">Preview flowchart akan muncul di sini.</div>';
    showBox($("renderError"), "");
    return;
  }

  try {
    showBox($("renderError"), "");
    const id = "mermaid-" + Date.now() + "-" + (++renderCounter);
    const result = await mermaid.render(id, trimmed);
    $("diagramBox").innerHTML = result.svg;
    if (typeof result.bindFunctions === "function") result.bindFunctions($("diagramBox"));
  } catch (error) {
    showBox($("renderError"), error.message || String(error));
  }
}

$("generateButton").addEventListener("click", async () => {
  const prompt = $("promptBox").value.trim();
  if (!prompt) return;

  if (generationPoll) clearInterval(generationPoll);
  $("generateButton").disabled = true;
  setRuntime("Membuat diagram", "busy");
  setBar($("generationProgress"), 5);
  $("generationStatus").textContent = "Mengirim prompt…";

  try {
    const result = await fetchJson("/api/generation/start", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ prompt })
    });

    generationPoll = setInterval(async () => {
      try {
        const status = await fetchJson("/api/generation/" + result.job_id, { cache: "no-store" });
        setBar($("generationProgress"), status.progress);
        $("generationStatus").textContent = [
          `Status: ${status.status}`,
          `Pesan: ${status.message}`,
          status.warning ? `Peringatan: ${status.warning}` : ""
        ].filter(Boolean).join("\n");

        if (status.status === "done") {
          clearInterval(generationPoll);
          generationPoll = null;
          $("generateButton").disabled = false;
          $("codeEditor").value = status.mermaid_code || "";
          await renderMermaid($("codeEditor").value);
          setRuntime("Siap", "ok");
        } else if (status.status === "error") {
          clearInterval(generationPoll);
          generationPoll = null;
          $("generateButton").disabled = false;
          showBox($("renderError"), status.error || status.message);
          setRuntime("Error", "error");
        }
      } catch (error) {
        $("generationStatus").textContent = error.message;
      }
    }, 1500);
  } catch (error) {
    $("generateButton").disabled = false;
    $("generationStatus").textContent = error.message;
    setRuntime("Error", "error");
  }
});

$("renderButton").addEventListener("click", () => renderMermaid($("codeEditor").value));

function downloadBlob(filename, blob) {
  const url = URL.createObjectURL(blob);
  const link = document.createElement("a");
  link.href = url;
  link.download = filename;
  document.body.appendChild(link);
  link.click();
  link.remove();
  URL.revokeObjectURL(url);
}

function currentSvgText() {
  const svg = $("diagramBox").querySelector("svg");
  if (!svg) throw new Error("Belum ada SVG yang dirender.");
  return new XMLSerializer().serializeToString(svg);
}

$("downloadSvgButton").addEventListener("click", () => {
  try {
    downloadBlob("mermaid-flowchart.svg", new Blob([currentSvgText()], {type:"image/svg+xml;charset=utf-8"}));
  } catch (error) {
    showBox($("renderError"), error.message);
  }
});

$("downloadPngButton").addEventListener("click", () => {
  try {
    const svg = $("diagramBox").querySelector("svg");
    const svgText = currentSvgText();
    const bounds = svg.getBoundingClientRect();
    const width = Math.max(800, Math.ceil(bounds.width || 1200));
    const height = Math.max(500, Math.ceil(bounds.height || 800));
    const scale = 2;
    const url = URL.createObjectURL(new Blob([svgText], {type:"image/svg+xml;charset=utf-8"}));
    const image = new Image();
    image.onload = () => {
      const canvas = document.createElement("canvas");
      canvas.width = width * scale;
      canvas.height = height * scale;
      const context = canvas.getContext("2d");
      context.fillStyle = "#fff";
      context.fillRect(0, 0, canvas.width, canvas.height);
      context.drawImage(image, 0, 0, canvas.width, canvas.height);
      URL.revokeObjectURL(url);
      canvas.toBlob((blob) => blob && downloadBlob("mermaid-flowchart.png", blob), "image/png");
    };
    image.src = url;
  } catch (error) {
    showBox($("renderError"), error.message);
  }
});

function formatBytes(bytes) {
  const value = Number(bytes) || 0;
  if (value < 1024) return value + " B";
  if (value < 1024 * 1024) return (value / 1024).toFixed(1) + " KB";
  return (value / (1024 * 1024)).toFixed(1) + " MB";
}

async function refreshDatasets(selectId=null) {
  const result = await fetchJson("/api/datasets", { cache: "no-store" });
  currentDatasets = result.datasets || [];

  const select = $("datasetSelect");
  select.innerHTML = '<option value="">Pilih dataset</option>';

  $("datasetList").innerHTML = currentDatasets.length
    ? ""
    : '<div class="placeholder">Belum ada dataset.</div>';

  currentDatasets.forEach((dataset) => {
    const option = document.createElement("option");
    option.value = dataset.dataset_id;
    option.textContent = `${dataset.name} — ${dataset.examples} contoh`;
    select.appendChild(option);

    const item = document.createElement("div");
    item.className = "dataset-item";
    item.innerHTML = `
      <div class="dataset-title"></div>
      <div class="muted"></div>
    `;
    item.querySelector(".dataset-title").textContent = dataset.name;
    item.querySelector(".muted").textContent =
      `${dataset.examples} contoh · ${formatBytes(dataset.size_bytes)}`;
    item.addEventListener("click", () => {
      select.value = dataset.dataset_id;
      showDatasetPreview(dataset.dataset_id);
    });
    $("datasetList").appendChild(item);
  });

  if (selectId && currentDatasets.some((item) => item.dataset_id === selectId)) {
    select.value = selectId;
    showDatasetPreview(selectId);
  }
}

async function showDatasetPreview(datasetId) {
  if (!datasetId) {
    $("datasetPreview").textContent = "Pilih dataset.";
    return;
  }
  try {
    const result = await fetchJson("/api/datasets/" + datasetId + "/preview", { cache:"no-store" });
    $("datasetPreview").textContent = JSON.stringify({ validation_report: result.validation_report, preview: result.preview }, null, 2);
  } catch (error) {
    $("datasetPreview").textContent = error.message;
  }
}

$("datasetSelect").addEventListener("change", (event) => showDatasetPreview(event.target.value));
$("refreshDatasetsButton").addEventListener("click", () => refreshDatasets());

$("uploadDatasetButton").addEventListener("click", async () => {
  const file = $("datasetFile").files[0];
  if (!file) {
    showBox($("uploadMessage"), "Pilih file .json atau .jsonl terlebih dahulu.");
    return;
  }

  const form = new FormData();
  form.append("dataset", file);
  $("uploadDatasetButton").disabled = true;
  $("uploadMessage").className = "notice";
  $("uploadMessage").textContent = "Mengunggah dan memvalidasi dataset…";

  try {
    const result = await fetchJson("/api/datasets/upload", {
      method: "POST",
      body: form
    });
    $("uploadMessage").className = "success";
    $("uploadMessage").textContent =
      `Dataset valid: ${result.dataset.name}\n` +
      `${result.dataset.examples} contoh · ${formatBytes(result.dataset.size_bytes)}\n` +
      `Status: ${result.dataset.validation_report.status} · Duplikat: ${result.dataset.validation_report.duplicate_examples} · Mermaid: ${(100 * result.dataset.validation_report.mermaid_ratio).toFixed(1)}%` +
      (result.dataset.validation_report.warnings.length ? `\nPeringatan: ${result.dataset.validation_report.warnings.join(" | ")}` : "");
    await refreshDatasets(result.dataset.dataset_id);
  } catch (error) {
    $("uploadMessage").className = "error";
    $("uploadMessage").textContent = error.message;
  } finally {
    $("uploadDatasetButton").disabled = false;
  }
});

function updateModeNotice() {
  const mode = $("trainingMode").value;
  if (mode === "lora") {
    $("modeNotice").textContent = "LoRA memuat ulang model dasar dalam FP16/BF16 dan melatih adapter.";
    $("learningRate").value = "0.0002";
  } else if (mode === "qlora") {
    $("modeNotice").textContent = "QLoRA memuat ulang model dasar 4-bit NF4. Memerlukan GPU CUDA dan menghemat VRAM.";
    $("learningRate").value = "0.0002";
  } else {
    $("modeNotice").textContent = "Full fine-tuning melatih seluruh parameter dan membutuhkan VRAM jauh lebih besar.";
    $("learningRate").value = "0.00002";
  }
}
$("trainingMode").addEventListener("change", updateModeNotice);

function collectTrainingConfig() {
  return {
    dataset_id: $("datasetSelect").value,
    mode: $("trainingMode").value,
    adapter_name: $("adapterName").value,
    epochs: $("epochs").value,
    max_steps: $("maxSteps").value,
    max_seq_length: $("maxSeqLength").value,
    batch_size: $("batchSize").value,
    gradient_accumulation_steps: $("gradientAccumulation").value,
    learning_rate: $("learningRate").value,
    lora_r: $("loraR").value,
    lora_alpha: $("loraAlpha").value,
    lora_dropout: $("loraDropout").value,
    warmup_ratio: $("warmupRatio").value,
    weight_decay: $("weightDecay").value,
    seed: $("seed").value,
    validation_ratio: $("validationRatio").value,
    logging_steps: $("loggingSteps").value,
    save_steps: $("saveSteps").value
  };
}

$("startTrainingButton").addEventListener("click", async () => {
  showBox($("trainingError"), "");
  showBox($("trainingSuccess"), "");
  $("trainingLog").value = "";

  if ($("trainingMode").value === "full") {
    const confirmed = window.confirm(
      "Full fine-tuning membutuhkan VRAM dan ruang disk jauh lebih besar. Lanjutkan?"
    );
    if (!confirmed) return;
  }

  try {
    const result = await fetchJson("/api/training/start", {
      method: "POST",
      headers: {"Content-Type":"application/json"},
      body: JSON.stringify(collectTrainingConfig())
    });

    activeTrainingJobId = result.job_id;
    $("startTrainingButton").disabled = true;
    $("cancelTrainingButton").disabled = false;
    $("downloadAdapterButton").disabled = true;
    setRuntime("Training", "busy");
    startTrainingPolling(result.job_id);
  } catch (error) {
    showBox($("trainingError"), error.message);
  }
});

function startTrainingPolling(jobId) {
  if (trainingPoll) clearInterval(trainingPoll);

  trainingPoll = setInterval(async () => {
    try {
      const status = await fetchJson("/api/training/" + jobId, { cache:"no-store" });
      setBar($("trainingProgress"), status.progress);
      $("trainingStatus").textContent = [
        `Status: ${status.status}`,
        `Pesan: ${status.message}`,
        status.total_steps ? `Langkah: ${status.current_step || 0}/${status.total_steps}` : "",
        status.epoch != null ? `Epoch: ${Number(status.epoch).toFixed(3)}` : "",
        status.latest_metrics ? `Metrik: ${JSON.stringify(status.latest_metrics)}` : ""
      ].filter(Boolean).join("\n");
      $("trainingLog").value = (status.logs || []).join("\n");
      $("trainingLog").scrollTop = $("trainingLog").scrollHeight;
      $("cancelTrainingButton").disabled =
        !["queued", "preparing", "training"].includes(status.status);

      if (["done", "error", "cancelled"].includes(status.status)) {
        clearInterval(trainingPoll);
        trainingPoll = null;
        $("startTrainingButton").disabled = false;
        $("cancelTrainingButton").disabled = true;

        if (status.status === "done") {
          showBox(
            $("trainingSuccess"),
            "Fine-tuning selesai. Hasil telah menjadi model aktif untuk generator Mermaid."
          );
          $("downloadAdapterButton").disabled = !status.downloadable;
          $("downloadAdapterButton").dataset.jobId = jobId;
          setRuntime("Siap", "ok");
          await refreshSystemStatus();
        } else if (status.status === "error") {
          showBox($("trainingError"), status.error || status.message);
          setRuntime("Error", "error");
        } else {
          showBox($("trainingError"), "Fine-tuning dibatalkan.");
          setRuntime("Siap", "ok");
        }
      }
    } catch (error) {
      $("trainingStatus").textContent = error.message;
    }
  }, 1500);
}

$("cancelTrainingButton").addEventListener("click", async () => {
  if (!activeTrainingJobId) return;
  try {
    await fetchJson("/api/training/" + activeTrainingJobId + "/cancel", {
      method: "POST"
    });
    $("cancelTrainingButton").disabled = true;
    $("trainingStatus").textContent += "\nPermintaan pembatalan dikirim.";
  } catch (error) {
    showBox($("trainingError"), error.message);
  }
});

$("downloadAdapterButton").addEventListener("click", () => {
  const jobId = $("downloadAdapterButton").dataset.jobId;
  if (jobId) window.location.href = "/api/training/" + jobId + "/download";
});

async function resumeLatestTraining() {
  try {
    const result = await fetchJson("/api/training", { cache: "no-store" });
    const latest = (result.jobs || [])[0];
    if (!latest) return;

    activeTrainingJobId = latest.job_id;

    if (["queued", "preparing", "training", "saving"].includes(latest.status)) {
      $("startTrainingButton").disabled = true;
      $("cancelTrainingButton").disabled =
        !["queued", "preparing", "training"].includes(latest.status);
      setRuntime("Training", "busy");
      startTrainingPolling(latest.job_id);
      return;
    }

    const status = await fetchJson(
      "/api/training/" + latest.job_id,
      { cache: "no-store" }
    );
    setBar($("trainingProgress"), status.progress);
    $("trainingStatus").textContent = [
      `Status terakhir: ${status.status}`,
      `Pesan: ${status.message}`
    ].join("\n");
    $("trainingLog").value = (status.logs || []).join("\n");

    if (status.status === "done" && status.downloadable) {
      $("downloadAdapterButton").disabled = false;
      $("downloadAdapterButton").dataset.jobId = latest.job_id;
    }
  } catch (error) {
    console.warn("Tidak dapat memulihkan status training:", error);
  }
}

await refreshDatasets();
await refreshSystemStatus();
await resumeLatestTraining();
setInterval(refreshSystemStatus, 10000);
</script>
</body>
</html>
"""


@app.errorhandler(413)
def upload_too_large(error):
    return jsonify(
        success=False,
        error="Ukuran unggahan melebihi batas 100 MB.",
    ), 413


@app.route("/")
def index():
    return render_template_string(
        PAGE,
        default_prompt=DEFAULT_FLOWCHART_REQUEST,
        mermaid_version=MERMAID_VERSION,
    )


@app.route("/api/system")
def api_system():
    gpu = {
        "available": torch.cuda.is_available(),
        "name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "allocated_bytes": (
            int(torch.cuda.memory_allocated())
            if torch.cuda.is_available()
            else 0
        ),
        "reserved_bytes": (
            int(torch.cuda.memory_reserved())
            if torch.cuda.is_available()
            else 0
        ),
    }

    return jsonify(
        success=True,
        model_runtime_state=(
            dict(MODEL_RUNTIME_STATE)
            if "MODEL_RUNTIME_STATE" in globals()
            else {"model_id": MODEL_ID, "mode": "base"}
        ),
        gpu=gpu,
        disk={
            "workspace": str(WEB_WORK_DIR),
            "free_bytes": int(shutil.disk_usage(WEB_WORK_DIR).free),
            "total_bytes": int(shutil.disk_usage(WEB_WORK_DIR).total),
        },
        training_running=training_is_running(),
        datasets=len(list_dataset_metadata()),
    )


@app.route("/api/datasets")
def api_datasets():
    public_items = []

    for metadata in list_dataset_metadata():
        public_items.append(
            {
                key: metadata.get(key)
                for key in (
                    "dataset_id",
                    "name",
                    "canonical_name",
                    "examples",
                    "size_bytes",
                    "uploaded_at",
                    "validation_report",
                )
            }
        )

    return jsonify(success=True, datasets=public_items)


@app.route("/api/datasets/upload", methods=["POST"])
def api_dataset_upload():
    uploaded = request.files.get("dataset")

    if uploaded is None or not uploaded.filename:
        return jsonify(
            success=False,
            error="Berkas dataset tidak ditemukan pada unggahan.",
        ), 400

    filename = secure_filename(uploaded.filename)
    if not filename:
        return jsonify(success=False, error="Nama file tidak valid."), 400

    try:
        metadata = save_uploaded_dataset(uploaded, filename)
        public_metadata = {
            key: metadata.get(key)
            for key in (
                "dataset_id",
                "name",
                "canonical_name",
                "examples",
                "size_bytes",
                "uploaded_at",
                "preview",
                "validation_report",
            )
        }
        return jsonify(success=True, dataset=public_metadata)
    except Exception as exc:
        return jsonify(success=False, error=str(exc)), 400


@app.route("/api/datasets/<dataset_id>/preview")
def api_dataset_preview(dataset_id):
    metadata = get_dataset_metadata(dataset_id)
    if metadata is None:
        return jsonify(success=False, error="Dataset tidak ditemukan."), 404

    return jsonify(
        success=True,
        dataset_id=dataset_id,
        name=metadata.get("name"),
        examples=metadata.get("examples"),
        preview=metadata.get("preview", []),
        validation_report=metadata.get("validation_report", {}),
    )


@app.route("/api/generation/start", methods=["POST"])
def api_generation_start():
    if training_is_running():
        return jsonify(
            success=False,
            error="Fine-tuning sedang berjalan; generasi dinonaktifkan sementara.",
        ), 409

    payload = request.get_json(silent=True) or {}
    prompt = str(payload.get("prompt") or "").strip()

    if not prompt:
        return jsonify(success=False, error="Prompt tidak boleh kosong."), 400

    job_id = uuid.uuid4().hex

    with generation_jobs_lock:
        generation_jobs[job_id] = {
            "status": "queued",
            "progress": 5,
            "message": "Pekerjaan generasi masuk antrean.",
            "prompt": prompt,
            "mermaid_code": "",
            "raw_response": "",
            "warning": None,
            "error": None,
            "created_at": time.time(),
        }

    threading.Thread(
        target=process_generation_job,
        args=(job_id, prompt),
        daemon=True,
    ).start()

    return jsonify(
        success=True,
        job_id=job_id,
        progress=5,
        message="Generasi dimulai.",
    )


@app.route("/api/generation/<job_id>")
def api_generation_status(job_id):
    job = get_generation_job(job_id)
    if job is None:
        return jsonify(success=False, error="Job generasi tidak ditemukan."), 404
    return jsonify(success=True, job_id=job_id, **job)


@app.route("/api/training")
def api_training_list():
    with training_jobs_lock:
        items = []
        for job_id, job in training_jobs.items():
            item = {
                "job_id": job_id,
                "status": job.get("status"),
                "progress": job.get("progress"),
                "message": job.get("message"),
                "created_at": job.get("created_at"),
                "updated_at": job.get("updated_at"),
                "downloadable": job.get("downloadable", False),
                "adapter_name": job.get("adapter_name"),
            }
            items.append(item)

    items.sort(
        key=lambda item: item.get("created_at") or 0,
        reverse=True,
    )
    return jsonify(success=True, jobs=items[:20])


@app.route("/api/training/start", methods=["POST"])
def api_training_start():
    global active_training_job_id

    try:
        config = parse_training_config(request.get_json(silent=True) or {})
    except Exception as exc:
        return jsonify(success=False, error=str(exc)), 400

    job_id = uuid.uuid4().hex

    with active_training_lock:
        if active_training_job_id is not None or training_is_running():
            return jsonify(
                success=False,
                error="Fine-tuning lain sedang berjalan.",
            ), 409
        active_training_job_id = job_id

    with training_jobs_lock:
        training_jobs[job_id] = {
            "status": "queued",
            "progress": 1,
            "message": "Fine-tuning masuk antrean.",
            "configuration": config,
            "logs": [],
            "latest_metrics": {},
            "cancel_requested": False,
            "current_step": 0,
            "total_steps": None,
            "epoch": None,
            "error": None,
            "downloadable": False,
            "created_at": time.time(),
            "updated_at": time.time(),
        }

    try:
        threading.Thread(
            target=run_training_job,
            args=(job_id, config),
            daemon=True,
        ).start()
    except Exception:
        with active_training_lock:
            if active_training_job_id == job_id:
                active_training_job_id = None
        raise

    return jsonify(
        success=True,
        job_id=job_id,
        message="Fine-tuning dimulai.",
    )


@app.route("/api/training/<job_id>")
def api_training_status(job_id):
    job = get_training_job(job_id)
    if job is None:
        return jsonify(success=False, error="Job training tidak ditemukan."), 404

    # Jangan membocorkan path internal selain pada unduhan terkontrol.
    public_job = dict(job)
    public_job.pop("output_dir", None)

    return jsonify(success=True, job_id=job_id, **public_job)


@app.route("/api/training/<job_id>/cancel", methods=["POST"])
def api_training_cancel(job_id):
    job = get_training_job(job_id)
    if job is None:
        return jsonify(success=False, error="Job training tidak ditemukan."), 404

    if job.get("status") not in {"queued", "preparing", "training"}:
        return jsonify(
            success=False,
            error="Job tidak berada pada status yang dapat dibatalkan.",
        ), 409

    update_training_job(
        job_id,
        cancel_requested=True,
        message="Permintaan pembatalan diterima.",
    )
    return jsonify(success=True, job_id=job_id)


@app.route("/api/training/<job_id>/download")
def api_training_download(job_id):
    job = get_training_job(job_id)
    if job is None:
        return jsonify(success=False, error="Job training tidak ditemukan."), 404

    if job.get("status") != "done" or not job.get("output_dir"):
        return jsonify(
            success=False,
            error="Hasil training belum tersedia.",
        ), 409

    output_dir = Path(job["output_dir"])
    if not output_dir.exists():
        return jsonify(success=False, error="Direktori hasil tidak ditemukan."), 404

    archive_base = WEB_WORK_DIR / f"{job_id}-{job.get('adapter_name', 'training-result')}"
    archive_path = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=str(output_dir),
        )
    )

    return send_file(
        archive_path,
        as_attachment=True,
        download_name=archive_path.name,
    )


@app.route("/health")
def health():
    return jsonify(
        ok=True,
        port=APP_PORT,
        model_available="qwen_model" in globals(),
        tokenizer_available="qwen_tokenizer" in globals(),
        training_running=training_is_running(),
    )


# ============================================================
# Server and Cloudflare tunnel
# ============================================================

class ServerThread(threading.Thread):
    def __init__(self, flask_app, host="0.0.0.0", port=7860):
        super().__init__(daemon=True)
        self.server = make_server(host, port, flask_app, threaded=True)
        self.ctx = flask_app.app_context()
        self.ctx.push()

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()


def stop_process_on_port(port):
    try:
        result = subprocess.run(
            ["bash", "-lc", f"fuser {port}/tcp 2>/dev/null || true"],
            capture_output=True,
            text=True,
        )

        current_pid = os.getpid()
        for item in result.stdout.strip().split():
            try:
                pid = int(item)
                if pid != current_pid:
                    os.kill(pid, signal.SIGTERM)
            except Exception:
                pass

        time.sleep(1)
    except Exception:
        pass


def install_cloudflared_if_needed():
    if shutil.which("cloudflared"):
        return

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            "/tmp/cloudflared.deb",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        check=True,
    )

    result = subprocess.run(
        ["dpkg", "-i", "/tmp/cloudflared.deb"],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        subprocess.run(["apt-get", "-f", "install", "-y"], check=True)


def start_cloudflare_tunnel(port):
    install_cloudflared_if_needed()

    try:
        if mermaid_cloudflare_proc.poll() is None:
            mermaid_cloudflare_proc.terminate()
    except Exception:
        pass

    proc = subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            f"http://127.0.0.1:{port}",
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    url_box = {"url": None}
    ready = threading.Event()

    def reader():
        with open(CLOUDFLARED_LOG_PATH, "w", encoding="utf-8") as log:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                match = re.search(
                    r"https://[-a-z0-9]+\.trycloudflare\.com",
                    line,
                )
                if match and url_box["url"] is None:
                    url_box["url"] = match.group(0)
                    ready.set()

    threading.Thread(target=reader, daemon=True).start()
    ready.wait(timeout=60)

    return proc, url_box["url"]


try:
    mermaid_app_server.shutdown()
except Exception:
    pass

stop_process_on_port(APP_PORT)

mermaid_app_server = ServerThread(app, host="0.0.0.0", port=APP_PORT)
mermaid_app_server.start()

ready = False
for _ in range(30):
    try:
        response = requests.get(
            f"http://127.0.0.1:{APP_PORT}/health",
            timeout=2,
        )
        if response.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    raise RuntimeError("Aplikasi web gagal dimulai.")

print("Aplikasi lokal:")
print(f"  http://127.0.0.1:{APP_PORT}")
print("Workspace:")
print(f"  {WEB_WORK_DIR}")
print("Model aktif:")
print(f"  {MODEL_RUNTIME_STATE}")

mermaid_cloudflare_proc, mermaid_public_url = start_cloudflare_tunnel(APP_PORT)

if mermaid_public_url:
    print("URL publik Cloudflare:")
    print(f"  {mermaid_public_url}")
else:
    print("URL publik belum terdeteksi. Periksa log:")
    print(f"  {CLOUDFLARED_LOG_PATH}")


INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:50:49] "GET /health HTTP/1.1" 200 -


Aplikasi lokal:
  http://127.0.0.1:7860
Workspace:
  /content/qwen_mermaid_web
Model aktif:
  {'model_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'mode': 'base', 'adapter_name': None, 'output_dir': None, 'updated_at': 1783911046.5840547}
[cloudflared] 2026-07-13T02:50:49Z INF Initiating graceful shutdown due to signal terminated ...
[cloudflared] 2026-07-13T02:50:49Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-07-13T02:50:49Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-07-13T02:50:49Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-07-13T02:50:49Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-07-13T02:50:49Z ERR Connection terminated connIndex=0
[cloudflared] 2026

INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:06] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:07] "GET /api/datasets HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:07] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:07] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:08] "GET /api/training HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:18] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:28] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:38] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:48] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:51:58] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:52:08] "GET /api/system HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:52:18] "GET /api/system HTTP/1.1" 200